# Travaux pratiques — Tokenisation et préparation du texte pour les modèles de langage

**Chapitre 3 : Tokenisation**

Dans ce TP, vous passerez d’un tokenizer simple fondé sur des règles à un tokenizer utilisant l’encodage par paires d’octets (BPE), comme celui de GPT-2. Vous apprendrez à construire un vocabulaire, à convertir les tokens en identifiants numériques, à gérer les tokens inconnus et à préparer des exemples d’entraînement pour un modèle de langage.

## Objectifs d’apprentissage

À la fin de ce TP, vous serez capable de :

1. Lire un fichier texte et examiner son contenu.
2. Expliquer pourquoi la tokenisation ne consiste pas simplement à séparer le texte par les espaces.
3. Construire un tokenizer simple à l’aide d’expressions régulières.
4. Construire un vocabulaire et attribuer un ID unique à chaque token.
5. Implémenter les opérations d’encodage et de décodage.
6. Gérer les tokens inconnus et les tokens spéciaux.
7. Utiliser un tokenizer BPE préentraîné avec `tiktoken`.
8. Préparer des séquences d’entraînement à l’aide d’une fenêtre glissante.
9. Comprendre les embeddings des tokens et les embeddings positionnels.

## TP — Compléter le code

Certaines portions de code ont été remplacées par `___`, comme dans les notebooks de TP à compléter.

- Remplacez chaque `___` par l'instruction, l'expression ou le paramètre attendu.
- Exécutez les cellules dans l'ordre.
- Appuyez-vous sur les commentaires et les explications pour compléter les portions manquantes.
- Corrigez toute erreur avant de poursuivre.

**Important :** les cellules contenant `___` ne fonctionneront pas correctement avant d'être complétées.

## Avant de commencer

- Environnement recommandé : Jupyter Notebook, JupyterLab ou Google Colab.
- Bibliothèques Python utilisées dans la suite : `tiktoken` et `torch`.
- Ce TP utilise la nouvelle d’Edith Wharton *The Verdict*, enregistrée dans le fichier `the-verdict.txt`.
- Exécutez le notebook de haut en bas. Certaines cellules dépendent de variables créées dans les cellules précédentes.

La cellule de préparation suivante télécharge le fichier texte s’il n’est pas déjà présent dans le répertoire de travail. Une connexion Internet est nécessaire uniquement pour ce premier téléchargement.

## Partie 1 — Construire un tokenizer simple

Nous allons d’abord examiner le fichier source. Le nombre de caractères donne une idée de sa taille ; l’affichage des premiers caractères permet de vérifier que le fichier a été correctement chargé.

### Charger le texte d’exemple

La cellule suivante vérifie si le fichier `the-verdict.txt` est disponible. Si ce n’est pas le cas, elle tente de télécharger le texte pédagogique depuis le dépôt associé. Si votre environnement n’a pas accès à Internet, placez le fichier `the-verdict.txt` dans le répertoire de travail du notebook avant de continuer.

In [ ]:
from pathlib import Path
from urllib.request import urlretrieve

text_path = ___  # TODO: créer un objet Path pour le fichier
if not text_path.exists():
    source_url = "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch/main/ch02/01_main-chapter-code/the-verdict.txt"
    try:
        urlretrieve(source_url, text_path)
        print(f"Texte d’exemple téléchargé dans : {text_path.resolve()}")
    except Exception as exc:
        raise RuntimeError(
            "Impossible de télécharger le fichier the-verdict.txt. Place the file in the notebook's "
            "working directory and run this cell again."
        ) from exc
else:
    print(f"Using existing text file: {text_path.resolve()}")

In [ ]:
# Lire tout le fichier texte UTF-8 dans une chaîne Python.
with open("the-verdict.txt", "r", encoding="utf-8")___ :
    raw_text = ___  # TODO: lire tout le contenu du fichier
    
print("Nombre total de caractères :", ...(raw_text))
print(___[:99])

Notre objectif est de découper le récit en tokens — principalement des mots et certains signes de ponctuation. Ces tokens pourront ensuite être associés à des identifiants entiers (*Token IDs*), qui constituent les entrées numériques utilisées par les modèles de langage.

Ce premier tokenizer est volontairement simple. Il illustre les idées principales, mais ne constitue pas un tokenizer complet adapté à toutes les langues et à tous les formats de texte.

Les pipelines de traitement des grands modèles de langage peuvent traiter des millions de documents. Nous utilisons ici une nouvelle courte afin que chaque étape reste compréhensible et s’exécute rapidement sur un ordinateur ordinaire.

Nous allons utiliser le module Python d’expressions régulières `re` pour construire un tokenizer à partir de règles explicites. Il n’est pas nécessaire de mémoriser toute la syntaxe des expressions régulières : concentrez-vous sur l’effet des règles sur les tokens obtenus.

### 1.1 Séparer le texte à partir des espaces

Le motif `\s` correspond à un caractère d’espacement. Les parenthèses autour d’un motif indiquent à `re.split()` de conserver les séparateurs correspondants dans le résultat. Cela est utile lorsque nous voulons que la ponctuation ou les espaces apparaissent comme des éléments distincts.

In [ ]:
# re.split() sépare à chaque espace correspondant ; les parenthèses conservent les séparateurs.
import re

text = "Hello, world. This, is a test."
result = ___  # TODO: utiliser re.split() avec un motif capturant les espaces

print(____)

Les caractères d’espacement capturés sont conservés dans la liste. C’est intentionnel : les parenthèses de `r'(\s)'` préservent chaque séparateur correspondant.

### 1.2 Traiter certains signes de ponctuation comme des tokens distincts

Nous pouvons étendre le motif afin que les virgules et les points soient également reconnus comme séparateurs. Le groupe capturant conserve ces signes de ponctuation dans le résultat.

In [ ]:
# Séparer sur les virgules, les points et les espaces ; les groupes capturants conservent les séparateurs.
result = ___  # TODO: utiliser re.split() avec un motif capturant virgules, points et espaces

print(___)

Les virgules et les points apparaissent maintenant comme des éléments distincts de la liste. Remarquez que la règle ne traite que les signes de ponctuation explicitement indiqués dans le motif ; un tokenizer simple ne reconnaîtra pas les cas qui nécessitent des règles supplémentaires.

### 1.3 Supprimer les chaînes vides et les éléments ne contenant que des espaces

Après la séparation, la liste peut contenir des espaces ou des chaînes vides. La compréhension de liste suivante conserve uniquement les éléments qui contiennent au moins un caractère autre qu’un espace.

In [ ]:
# Conserver uniquement les éléments non vides (qui ne contiennent pas uniquement des espaces).
result = ___  # TODO: conserver les éléments non vides
print(____)

**Faut-il conserver les espaces comme tokens ?**

Cela dépend de la tâche. Pour un texte courant, il est souvent inutile de conserver les espaces comme tokens. En revanche, lorsque les espaces ont une signification — par exemple dans du code source ou de la poésie — il peut être important de les préserver. Les règles de tokenisation doivent être adaptées à l’application.

### 1.4 Étendre les règles à d’autres signes de ponctuation

Le motif suivant traite plusieurs signes de ponctuation ainsi que le double tiret (`--`). Il s’agit toujours d’un petit ensemble de règles écrites manuellement : il est utile pour apprendre, mais ne couvre pas tous les cas possibles.

In [ ]:
# Tester des règles de ponctuation plus larges, notamment le double tiret (--) comme un seul token.
____ = "Hello, world. Is this-- a test?"
____ = ___  # TODO: utiliser re.split() avec un motif capturant la ponctuation, -- et les espaces
____ = [item.___ for ___ in ___ if item.___()]
print(___)

In [ ]:
# Supprimer les espaces au début et à la fin de chaque élément, puis éliminer les chaînes vides.
# Conserver uniquement les éléments non vides (qui ne contiennent pas uniquement des espaces).
____ = [___ for ___ in ___ if ___.____()]
print(___)

In [ ]:
text = "Hello, world. Is this-- a test?"

___ = ___  # TODO: réutiliser le motif de tokenisation précédent
___ = [___.____() for ___ in ___ if ___.___()]
print(result)

### 1.5 Appliquer le tokenizer à l’ensemble du récit

Nous pouvons maintenant appliquer les mêmes règles à l’intégralité du texte. La liste obtenue est stockée dans la variable `preprocessed`.

In [ ]:
# Appliquer les mêmes règles de tokenisation à l’ensemble du récit.
preprocessed = ___  # TODO: appliquer re.split() et les règles au texte complet
______ = ___  # TODO: nettoyer et supprimer les éléments vides
print(____[:30])

In [ ]:
print(len(preprocessed))

## Partie 2 — Construire un vocabulaire et attribuer des Token IDs

Un modèle travaille avec des nombres plutôt qu’avec des chaînes de caractères brutes. Nous créons donc un vocabulaire, c’est-à-dire une association entre chaque token distinct et un identifiant entier unique.

La fonction `set()` élimine les tokens en double. Nous trions les tokens uniques pour obtenir un vocabulaire déterministe, puis nous attribuons un ID entier à chaque token.

In [ ]:
# set() élimine les doublons ; sorted() produit un ordre de vocabulaire reproductible.
all_words = ___  # TODO: obtenir les tokens uniques puis les trier
_____ = ___  # TODO: calculer le nombre de tokens uniques

print(vocab_size)

La taille du vocabulaire correspond au nombre de tokens uniques. La valeur exacte dépend du texte et des règles de tokenisation. La cellule suivante affiche les premières entrées pour illustrer l’association entre les tokens et leurs IDs.

In [ ]:
# Attribuer à chaque token unique un ID entier unique, en commençant par 0.
vocab = ___  # TODO: associer chaque token à un ID unique

In [ ]:
# Afficher un petit échantillon du vocabulaire plutôt que toutes les entrées.
for i, ____ in ____(vocab.items()):
    print(item)
    if i >= 50:
        ____

Chaque token est associé à un entier unique. Ces IDs sont des étiquettes arbitraires : leur ordre numérique ne signifie pas qu’un token est plus important qu’un autre.

L’encodage transforme les tokens du texte en IDs entiers. Le décodage nécessite l’association inverse, des IDs vers les chaînes de tokens. Nous allons conserver ces deux associations dans la classe du tokenizer.

### 2.1 Implémenter `SimpleTokenizerV1`

La classe possède deux méthodes principales :

- `encode(text)` : découper le texte en tokens et convertir chaque token en ID entier.
- `decode(ids)` : convertir les IDs en chaînes de tokens, puis les réunir pour reconstruire le texte.

Cette première version suppose que chaque token rencontré est présent dans le vocabulaire. Nous vérifierons cette hypothèse prochainement.

**Plan d’implémentation**

1. Stocker le vocabulaire qui associe chaque token à son ID.
2. Construire le vocabulaire inverse, qui associe chaque ID à un token.
3. Découper le texte d’entrée selon les mêmes règles que celles utilisées pour construire le vocabulaire.
4. Convertir chaque token en ID.
5. Décoder les IDs et ajuster les espaces avant certains signes de ponctuation.

In [ ]:
# La version 1 suppose que tous les tokens sont présents dans le vocabulaire.
class SimpleTokenizerV1:
    def __init__(self, vocab):
        # Conserver les deux associations pour permettre l’encodage et le décodage.
        self.str_to_int = ___ # TODO: construire le dictionnaire token -> ID
        self.int_to_str = ___  # TODO: construire le dictionnaire inverse ID -> token

    def encode(self, text):
        # Découper le texte selon les mêmes règles que celles utilisées pour construire le vocabulaire.
preprocessed = ___  # TODO: découper le texte avec les mêmes règles
        preprocessed = [item.strip() for item in preprocessed if item.strip()]

        # This version raises KeyError if a token is not in the vocabulary.
        ids = ___  # TODO: convertir chaque token en ID
        return ids

    def decode(self, ids):
        # Convert IDs back to token strings and join them with spaces.
        text = ___  # TODO: convertir les IDs en tokens puis les joindre

        # Remove spaces before common punctuation marks.
        text = re.___(r'\s+([,.?!"()\'])', r'\1', ___) 
        return text

### 2.2 Tester l’encodage et le décodage

Nous utilisons un court extrait du récit. L’encodage doit produire une liste d’entiers ; le décodage doit reconstruire une version lisible du texte d’entrée. Comme ce tokenizer réunit les tokens avec des espaces puis corrige l’espacement avant certains signes de ponctuation, il ne préservera pas nécessairement tous les détails de mise en forme d’origine.

In [ ]:
tokenizer = SimpleTokenizerV1(vocab)

text = """"It's the last he painted, you know," 
           Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.____(text)
print(__) 

La cellule suivante décode les IDs. Comparez le résultat au texte d’origine. L’objectif est de récupérer le contenu du texte, et non de prétendre que ce tokenizer écrit manuellement peut reproduire exactement toutes les configurations possibles d’espacement.

In [ ]:
tokenizer.____(ids) #This should return the original text (or a very close approximation, depending on tokenization).

Le cycle encodage/décodage fonctionne pour ce passage parce que ses tokens étaient présents dans le vocabulaire construit précédemment.

### 2.3 Que se passe-t-il avec un mot jamais rencontré ?

Un vocabulaire construit à partir d’un seul récit ne peut pas contenir tous les mots susceptibles d’apparaître par la suite. L’exemple suivant contient des mots absents du vocabulaire. Avec `SimpleTokenizerV1`, un token inconnu provoque une erreur `KeyError`.

In [ ]:
text = "Hello, do you like tea?"
print(tokenizer.___(__))    #This should return the token IDs for the input text.

C’est le problème des **mots hors vocabulaire** (*Out-Of-Vocabulary*, OOV). Un tokenizer doit définir le comportement à adopter lorsqu’il rencontre un token absent de son vocabulaire. Une solution simple consiste à remplacer tout token inconnu par un token spécial tel que `<|unk|>`.

## Partie 3 — Gérer les tokens inconnus et les tokens spéciaux

Nous créons maintenant `SimpleTokenizerV2`, qui prend en charge deux tokens spéciaux :

- `<|unk|>` représente un token absent du vocabulaire.
- `<|endoftext|>` peut marquer une frontière entre plusieurs documents textuels.

Les tokens spéciaux sont des entrées réservées du vocabulaire auxquelles un rôle précis est attribué pendant le traitement ou l’entraînement.

Lorsqu’un mot est absent du vocabulaire, l’encodeur le remplacera par `<|unk|>` au lieu de provoquer une erreur. Nous ajoutons également `<|endoftext|>` afin de pouvoir réunir plusieurs échantillons de texte tout en conservant une frontière explicite entre les documents.

Ajoutez les tokens spéciaux avant de construire le nouveau vocabulaire. Cela garantit que chaque token spécial possède son propre ID entier.

In [ ]:
# Ajouter les tokens réservés avant de construire l’association token-ID mise à jour.
all_tokens = ___(list(___(preprocessed))) #TODO: obtenir les tokens uniques puis les trier
all_tokens.extend(___)  # TODO: ajouter les tokens spéciaux

vocab = ___  # TODO: construire le dictionnaire token -> ID

In [ ]:
len(___.___()) #TODO: vérifier le nombre de tokens dans le vocabulaire mis à jour

Le vocabulaire contient maintenant deux entrées supplémentaires. Sa taille augmente de deux, à condition que ces chaînes de tokens spéciaux n’aient pas déjà été présentes.

Examinez les dernières entrées du vocabulaire pour vérifier que les tokens spéciaux ont bien été ajoutés.

In [ ]:
# Examiner les dernières entrées pour vérifier la présence des tokens spéciaux.
for i, ___ in ____(list(___.___())[-5:]): #TODO: examiner les dernières entrées du vocabulaire
    print(___) #TODO: afficher l’ID et le token correspondant

### 3.1 Implémenter `SimpleTokenizerV2`

La deuxième version applique les mêmes règles de tokenisation, mais remplace tout token absent du vocabulaire par `<|unk|>`. Le décodage associe ensuite chaque ID à la chaîne de caractères correspondante.

In [ ]:
# La version 2 remplace les tokens absents du vocabulaire par <|unk|>.
class SimpleTokenizerV2:
    def __init__(self, vocab):
        # Conserver les associations token-ID et ID-token.
        self.str_to_int = ___  # TODO: construire le dictionnaire token -> ID
        self.int_to_str = {token_id: token for token, token_id in vocab.items()}

    def encode(self, text):
        # Split text using the same rules as the vocabulary-building step.
preprocessed = ___  # TODO: découper le texte avec les mêmes règles
        preprocessed = [item.___() for item in ___ if item.___()]

        # Replace each out-of-vocabulary token with the reserved <|unk|> token.
        preprocessed = [
            ___  # TODO: garder le token connu, sinon utiliser <|unk|>
            for token in ___ #TODO: parcourir les tokens prétraités
        ]
        return ___  # TODO: convertir les tokens en IDs

    def decode(self, ids):
        # Map each integer ID back to its token string.
        text = " ".___([self.int_to_str[token_id] for ____ in ids])

        # Remove spaces before common punctuation marks.
        text = re.____(r'\s+([,.:;?!"()\'])', r'\1', ____)
        return ___

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

text = " <|endoftext|> ".____((___, ___)) #TODO: utiliser la méthode encode() pour obtenir les IDs des deux textes, puis les concaténer avec le token <|endoftext|> entre eux

print(text)

In [ ]:
tokenizer.____(text) #This should return the token IDs for the concatenated input text.

In [ ]:
tokenizer.___(tokenizer.___(text)) # This should return the original text (or a very close approximation, depending on tokenization).

Le texte décodé indique où se trouvent les tokens inconnus. Un seul `<|unk|>` ne conserve pas l’orthographe du mot original jamais rencontré ; il indique uniquement que ce mot ne figurait pas dans le vocabulaire.

### Autres tokens spéciaux

Selon le modèle et la configuration d’entraînement, les tokenizers peuvent utiliser d’autres marqueurs :

- **BOS** (*Beginning Of Sequence*) : marque le début d’une séquence.
- **EOS** (*End Of Sequence*) : marque la fin d’une séquence.
- **PAD** (*Padding*) : complète les séquences plus courtes afin que les exemples d’un même lot aient la même longueur.

Les différentes familles de modèles utilisent des conventions différentes. Ces noms désignent des concepts et non des chaînes de tokens universelles.

Les tokenizers des modèles de la famille GPT n’utilisent pas tous les mêmes tokens spéciaux. Dans l’encodage GPT-2 présenté plus loin, `<|endoftext|>` est un token spécial important. Ne supposez pas que tous les modèles utilisent BOS, EOS, PAD ou UNK exactement de la même manière.

Le tokenizer de GPT-2 n’utilise pas `<|unk|>` pour les mots ordinaires jamais rencontrés. Son encodage par paires d’octets (BPE) peut représenter le texte à l’aide de sous-mots ou d’unités au niveau des octets, ce qui permet d’encoder de nombreuses chaînes qui n’ont pas été rencontrées comme mots complets pendant l’entraînement.

## Partie 4 — Encodage par paires d’octets (BPE) avec `tiktoken`

Notre tokenizer personnalisé permet de comprendre les vocabulaires et les IDs, mais il repose sur des tokens entiers et sur un petit ensemble de règles de ponctuation. Les tokenizers modernes de LLM utilisent couramment des méthodes par sous-mots telles que BPE.

Nous allons utiliser la bibliothèque `tiktoken` avec l’encodage GPT-2 pour examiner comment un tokenizer préentraîné transforme le texte en IDs.

### 4.1 Installer et charger `tiktoken`

In [ ]:
%pip install -q tiktoken

In [ ]:
import importlib.metadata
import tiktoken

print("tiktoken version:", importlib.metadata.version("tiktoken"))

In [ ]:
# Charger l’encodage BPE préentraîné de GPT-2 avec tiktoken.
tokenizer = ___  # TODO: charger l’encodage GPT-2

In [ ]:
text = (
    "Hello, do you like tea? <|endoftext|> In the sunlit terraces"
     " of someunknownPlace."
)

integers = ___  # TODO: encoder le texte en autorisant le token spécial

print(integers)

In [ ]:
# Décoder les Token IDs pour vérifier que le texte d’origine peut être reconstruit.
strings = tokenizer.___(integers) #TODO: décoder les IDs en texte

print(strings)

### Exercice 1 — Examiner la tokenisation BPE

Encodez la chaîne ci-dessous, qui peut sembler inhabituelle, puis décodez-la. Le tokenizer doit-il posséder une entrée correspondant à chaque mot complet ? Examinez la liste des IDs et le résultat du cycle encodage/décodage.

In [ ]:
# Décoder les Token IDs pour vérifier que le texte d’origine peut être reconstruit.
integers = ___  # TODO: encoder cette chaîne
print(___)

strings = ___  # TODO: décoder la liste d’IDs
print(___)

## Partie 5 — Préparer des exemples d’entraînement avec une fenêtre glissante

Les modèles de langage sont entraînés à prédire le token suivant. Pour cette raison, chaque séquence d’entrée est associée à une séquence cible décalée d’une position vers la droite.

In [ ]:
# Lire tout le fichier texte UTF-8 dans une chaîne Python.
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = ___  # TODO: encoder le texte complet
print(len(____))  #TODO: afficher le nombre de tokens dans le texte complet

In [ ]:
enc_sample = ___[50:] #TODO: encoder le texte complet

In [ ]:
context_size = 4

x = ___  # TODO: prendre les premiers IDs du contexte
y = ___  # TODO: décaler la séquence d’un token

print(f"x: {x}")
print(f"y:      {y}")

In [ ]:
# Afficher des exemples de prédiction du token suivant avec des contextes de longueur croissante.
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(context, "---->", desired)

In [ ]:
# Afficher des exemples de prédiction du token suivant avec des contextes de longueur croissante.
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]

    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

### 5.1 Construire un jeu de données PyTorch et un chargeur de données

Un jeu de données (*Dataset*) crée des séquences d’entrée et des séquences cibles de longueur fixe à partir des IDs des tokens. Un chargeur de données (*DataLoader*) regroupe ces exemples en lots (*batches*), ce qui facilite leur transmission au modèle.

In [ ]:
# PyTorch fournit Dataset pour représenter les exemples et DataLoader pour les regrouper en lots.
import torch
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokeniser l’ensemble du texte
        token_ids = ___  # TODO: convertir le texte en IDs

        # Utiliser une fenêtre glissante pour découper le texte en séquences chevauchantes de longueur max_length
        for i in ___:  # TODO: parcourir les IDs par fenêtres glissantes
            input_chunk = token_ids[i:i + max_length]
            target_chunk = ___  # TODO: créer la cible décalée d’un token
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

In [ ]:
def create_dataloader_v1(
    txt,
    batch_size=4,
    max_length=256,
    stride=128,
    shuffle=True,
    drop_last=True,
    num_workers=0,
):
    # Charger le tokenizer GPT-2.
    tokenizer = ___  # TODO: charger l’encodage GPT-2

    # Convertir le texte en exemples d’entrée et de cible de longueur fixe.
    dataset = ___  # TODO: construire le dataset

    # Regrouper les exemples en lots pour l’entraînement du modèle.
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers,
    )
    return dataloader

In [ ]:
# Lire tout le fichier texte UTF-8 dans une chaîne Python.
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
second_batch = next(data_iter)
print(second_batch)

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

## Partie 6 — Embeddings des tokens

Les Token IDs sont des étiquettes discrètes et non des mesures numériques porteuses de sens. Une couche d’embedding associe chaque ID à un vecteur dense dont les valeurs peuvent être apprises.

In [ ]:
# Exemples de Token IDs. Chaque ID doit appartenir à l’intervalle du vocabulaire défini ci-dessous.
input_ids = ___  # TODO: créer un tenseur contenant des IDs

In [ ]:
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = ___  # TODO: créer la couche d’embedding

In [ ]:
print(embedding_layer.weight)

In [ ]:
print(embedding_layer(torch.tensor([3])))

In [ ]:
print(embedding_layer(input_ids))

## Partie 7 — Embeddings positionnels

Les embeddings des tokens indiquent quels tokens sont présents, mais ne codent pas à eux seuls leur ordre. Les embeddings positionnels apportent une information sur la position de chaque token dans la séquence. Dans cet exemple, les embeddings des tokens et les embeddings positionnels ont la même dimension vectorielle ; ils peuvent donc être additionnés.

In [ ]:
vocab_size = 50257
output_dim = 256

token_embedding_layer = ___  # TODO: créer la couche d’embedding des tokens

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length,
    stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [ ]:
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape)

In [ ]:
# Convertir chaque Token ID en vecteur dense ; dimensions : lot × séquence × embedding.
token_embeddings = ___  # TODO: convertir les IDs en vecteurs
print(token_embeddings.shape)

In [ ]:
context_length = max_length
pos_embedding_layer = ___  # TODO: créer l’embedding positionnel

In [ ]:
pos_embeddings = pos_embedding_layer(torch.arange(max_length))
print(pos_embeddings.shape)

In [ ]:
# Ajouter l’information positionnelle aux embeddings des tokens (diffusée sur la dimension du lot).
input_embeddings = ___  # TODO: additionner embeddings des tokens et positions
print(input_embeddings.shape)

## Résumé du TP

- Un tokenizer transforme le texte en une séquence de tokens.
- Un vocabulaire associe chaque token à un ID entier.
- Un tokenizer simple au niveau des mots peut échouer sur les mots jamais rencontrés ; le token `<|unk|>` constitue une solution possible.
- Les tokenizers BPE peuvent découper le texte en sous-mots ou en unités au niveau des octets, de sorte qu’un mot complet inconnu peut malgré tout être encodé.
- Les exemples d’entraînement pour la prédiction du token suivant utilisent une séquence d’entrée et une séquence cible décalée d’un token.
- Les embeddings des tokens fournissent des vecteurs appris pour les IDs ; les embeddings positionnels indiquent la position des tokens dans la séquence.
- Le tokenizer, le vocabulaire et la préparation des séquences constituent des étapes importantes du pipeline d’un LLM.